# Gaussian Process Regression from Scratch

This notebook implements **Gaussian Process (GP) regression** from first principles using NumPy and SciPy.
It compares two kernels, samples from the prior, fits hyperparameters by marginal likelihood optimisation,
and visualises the posterior predictive distribution.

The project demonstrates:

- squared exponential and Laplace-style covariance kernels;
- sampling functions from a GP prior;
- negative log marginal likelihood optimisation;
- posterior mean, covariance and sample paths;
- comparison using negative log marginal likelihood and mean squared error.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

np.random.seed(42)


## Kernels

A GP is defined by a mean function and a covariance function. I use a zero mean function and compare
two covariance choices controlled by three hyperparameters:

- lengthscale $\ell$;
- signal variance $\sigma^2$;
- observation noise variance $\sigma_e^2$.


In [ ]:
def squared_exponential_kernel(x, y, theta):
    """Squared exponential / RBF covariance kernel."""
    lengthscale, variance, _ = theta
    x = np.atleast_1d(x)
    y = np.atleast_1d(y)
    distance2 = (x[:, None] - y[None, :]) ** 2
    return variance * np.exp(-distance2 / (2.0 * lengthscale**2))


def laplace_kernel(x, y, theta):
    """Laplace-style covariance kernel."""
    lengthscale, variance, _ = theta
    x = np.atleast_1d(x)
    y = np.atleast_1d(y)
    distance = np.abs(x[:, None] - y[None, :])
    return variance * np.exp(-distance / (2.0 * lengthscale**2))


def add_observation_noise(K, noise_variance):
    """Add independent Gaussian observation noise to a covariance matrix."""
    return K + noise_variance * np.eye(K.shape[0])


## GP regression class

The class below implements prior sampling, negative log marginal likelihood optimisation and posterior
prediction. Cholesky decompositions are used for numerical stability.


In [ ]:
class GaussianProcessRegressorFromScratch:
    """Minimal Gaussian Process regression implementation."""

    def __init__(self, kernel="SE", mean=0.0, jitter=1e-8):
        self.kernel_name = kernel
        self.mean = mean
        self.jitter = jitter

    def init_hypers(self, theta):
        self.theta = np.asarray(theta, dtype=float)

    def load_data(self, x, y):
        self.x_obs = np.asarray(x, dtype=float)
        self.y_obs = np.asarray(y, dtype=float)

    def set_space(self, x):
        self.space = np.asarray(x, dtype=float)

    def kernel(self, x, y, theta=None):
        theta = self.theta if theta is None else np.asarray(theta, dtype=float)
        if self.kernel_name == "SE":
            return squared_exponential_kernel(x, y, theta)
        if self.kernel_name == "Laplace":
            return laplace_kernel(x, y, theta)
        raise ValueError(f"Unknown kernel: {self.kernel_name}")

    def sample_from_prior(self, trajectories=5):
        K = self.kernel(self.space, self.space)
        L = np.linalg.cholesky(K + self.jitter * np.eye(len(self.space)))
        z = np.random.randn(len(self.space), trajectories)

        self.K_prior = K
        self.prior_samples = (self.mean + L @ z).T
        return self.prior_samples

    def negative_log_marginal_likelihood(self, theta):
        theta = np.asarray(theta, dtype=float)
        if np.any(theta <= 0):
            return np.inf

        K = self.kernel(self.x_obs, self.x_obs, theta)
        K = add_observation_noise(K, theta[2])
        K = K + self.jitter * np.eye(len(self.x_obs))

        L = np.linalg.cholesky(K)
        centred_y = self.y_obs - self.mean

        v = np.linalg.solve(L, centred_y)
        alpha = np.linalg.solve(L.T, v)

        data_fit = 0.5 * centred_y @ alpha
        complexity = np.sum(np.log(np.diag(L)))
        normalisation = 0.5 * len(self.x_obs) * np.log(2 * np.pi)
        return data_fit + complexity + normalisation

    def train(self, method="L-BFGS-B", disp=False):
        bounds = [(1e-8, None), (1e-8, None), (1e-8, None)]
        result = minimize(
            self.negative_log_marginal_likelihood,
            self.theta,
            method=method,
            bounds=None if method == "Nelder-Mead" else bounds,
            options={"maxiter": 500, "disp": disp},
        )
        self.theta = result.x
        self.final_loss = result.fun
        return result, result.fun

    def sample_from_posterior(self, trajectories=5):
        x = self.x_obs
        y = self.y_obs - self.mean
        x_star = self.space
        noise_variance = self.theta[2]

        K = self.kernel(x, x)
        K = add_observation_noise(K, noise_variance)
        K_star = self.kernel(x, x_star)
        K_starstar = self.kernel(x_star, x_star)

        L = np.linalg.cholesky(K + self.jitter * np.eye(len(x)))
        v = np.linalg.solve(L, y)
        alpha = np.linalg.solve(L.T, v)

        post_mean = self.mean + K_star.T @ alpha
        W = np.linalg.solve(L, K_star)
        post_cov = K_starstar - W.T @ W
        post_cov = 0.5 * (post_cov + post_cov.T)

        L_post = np.linalg.cholesky(post_cov + self.jitter * np.eye(len(x_star)))
        z = np.random.randn(len(x_star), trajectories)
        samples = (post_mean[:, None] + L_post @ z).T

        self.post_mean = post_mean
        self.post_cov = post_cov
        self.post_samples = samples
        return samples

    def plot_prior(self, title="GP prior"):
        x = self.space
        var = np.diag(self.K_prior)

        plt.figure(figsize=(12, 4))
        for sample in self.prior_samples:
            plt.plot(x, sample, alpha=0.55)

        plt.plot(x, np.zeros_like(x) + self.mean, lw=2, label="prior mean")
        plt.fill_between(x, self.mean - 2 * np.sqrt(var), self.mean + 2 * np.sqrt(var), alpha=0.2)
        plt.xlabel("x")
        plt.ylabel("f(x)")
        plt.title(title)
        plt.legend()
        plt.show()

    def plot_posterior(self, title="GP posterior"):
        x_star = self.space
        var = np.maximum(np.diag(self.post_cov), 0)

        plt.figure(figsize=(12, 4))
        for sample in self.post_samples:
            plt.plot(x_star, sample, alpha=0.45)

        plt.plot(x_star, self.post_mean, lw=2, label="posterior mean")
        plt.fill_between(
            x_star,
            self.post_mean - 2 * np.sqrt(var),
            self.post_mean + 2 * np.sqrt(var),
            alpha=0.25,
            label="approx. 95% interval",
        )
        plt.scatter(self.x_obs, self.y_obs, s=15, label="observations")
        plt.xlabel("x")
        plt.ylabel("f(x)")
        plt.title(title)
        plt.legend()
        plt.show()


## Sampling from the prior

Prior samples show how the kernel hyperparameters encode assumptions about smoothness and vertical scale
before observing data.


In [ ]:
x_grid = np.linspace(0, 1750, 100)

gp_se_prior = GaussianProcessRegressorFromScratch(kernel="SE")
gp_se_prior.init_hypers(theta=[200, 25, 1])
gp_se_prior.set_space(x_grid)
gp_se_prior.sample_from_prior(trajectories=5)
gp_se_prior.plot_prior(title="Prior samples: squared exponential kernel")

gp_laplace_prior = GaussianProcessRegressorFromScratch(kernel="Laplace")
gp_laplace_prior.init_hypers(theta=[200, 25, 1])
gp_laplace_prior.set_space(x_grid)
gp_laplace_prior.sample_from_prior(trajectories=5)
gp_laplace_prior.plot_prior(title="Prior samples: Laplace-style kernel")


In [ ]:
def compare_prior_lengthscales(kernel_name, lengthscales, variance=20, noise_variance=1):
    x_grid = np.linspace(0, 1750, 100)
    plt.figure(figsize=(12, 4))

    for lengthscale in lengthscales:
        gp = GaussianProcessRegressorFromScratch(kernel=kernel_name)
        gp.init_hypers(theta=[lengthscale, variance, noise_variance])
        gp.set_space(x_grid)
        gp.sample_from_prior(trajectories=1)
        plt.plot(x_grid, gp.prior_samples[0], label=f"ell={lengthscale}")

    plt.axhline(0, lw=1, label="prior mean")
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title(f"Effect of lengthscale: {kernel_name} kernel")
    plt.legend()
    plt.show()


compare_prior_lengthscales("SE", lengthscales=[50, 200, 600])
compare_prior_lengthscales("Laplace", lengthscales=[50, 200, 600])


## Data

The experiment uses the one-dimensional signal stored in `data/hr2.txt`. The series is mean-centred before fitting.

If `data/hr2.txt` is removed, the notebook falls back to a synthetic heart-rate-like signal so that the project still runs.

In [ ]:
def generate_synthetic_signal(n=1750, seed=42):
    """Generate a smooth noisy one-dimensional signal for GP regression demos."""
    rng = np.random.default_rng(seed)
    x = np.arange(n)
    slow_trend = 8 * np.sin(2 * np.pi * x / 600)
    medium_cycle = 3 * np.sin(2 * np.pi * x / 120)
    local_bump = 10 * np.exp(-0.5 * ((x - 950) / 90) ** 2)
    noise = rng.normal(scale=1.5, size=n)
    y = slow_trend + medium_cycle + local_bump + noise
    return y - np.mean(y)


data_path = Path("../data/hr2.txt")
if data_path.exists():
    y_data = np.loadtxt(data_path)
    y_data = y_data - np.mean(y_data)
    print(f"Loaded data from {data_path}")
else:
    y_data = generate_synthetic_signal()
    print("Using synthetic signal because data/hr2.txt was not found.")

x_data = np.arange(len(y_data))

rng = np.random.default_rng(42)
idx = rng.choice(len(y_data), size=max(20, len(y_data) // 10), replace=False)
idx.sort()

x_obs = x_data[idx]
y_obs = y_data[idx]

plt.figure(figsize=(12, 4))
plt.plot(x_data, y_data, label="full signal", alpha=0.65)
plt.scatter(x_obs, y_obs, s=12, label="observed subset")
plt.xlabel("time index")
plt.ylabel("centred signal")
plt.title("Observed subset used for GP regression")
plt.legend()
plt.show()


## Hyperparameter optimisation

The hyperparameters are chosen by minimising the negative log marginal likelihood. This balances data fit
against model complexity.


In [ ]:
def fit_gp(kernel_name, x_obs, y_obs, theta0=[200, 25, 1], method="L-BFGS-B"):
    gp = GaussianProcessRegressorFromScratch(kernel=kernel_name)
    gp.init_hypers(theta=theta0)
    gp.load_data(x_obs, y_obs)
    result, loss = gp.train(method=method, disp=False)
    print(f"{kernel_name:7s} | NLL={loss:.2f} | theta={np.round(gp.theta, 3)}")
    return gp, result, loss


gp_se, result_se, nll_se = fit_gp("SE", x_obs, y_obs)
gp_laplace, result_laplace, nll_laplace = fit_gp("Laplace", x_obs, y_obs)


In [ ]:
initial_thetas = [
    [50.0, 25.0, 1.0],
    [200.0, 25.0, 1.0],
    [600.0, 25.0, 1.0],
    [200.0, 5.0, 1.0],
    [200.0, 50.0, 1.0],
    [200.0, 25.0, 0.1],
    [200.0, 25.0, 10.0],
]


def compare_initialisations(kernel_name, initial_thetas):
    rows = []
    for theta0 in initial_thetas:
        gp, _, loss = fit_gp(kernel_name, x_obs, y_obs, theta0=theta0)
        rows.append({"kernel": kernel_name, "theta0": theta0, "theta_hat": gp.theta, "nll": loss})
    return rows


results_initialisation = []
for kernel_name in ["SE", "Laplace"]:
    results_initialisation.extend(compare_initialisations(kernel_name, initial_thetas))


## Posterior prediction

After fitting the hyperparameters, the GP posterior gives a predictive mean and uncertainty band over the
full input grid. For speed, posterior sample paths are drawn on a moderately sized prediction grid.


In [ ]:
prediction_grid = np.linspace(0, len(y_data) - 1, 600)

gp_se.set_space(prediction_grid)
gp_se.sample_from_posterior(trajectories=5)
gp_se.plot_posterior(title="Posterior prediction: squared exponential kernel")

gp_laplace.set_space(prediction_grid)
gp_laplace.sample_from_posterior(trajectories=5)
gp_laplace.plot_posterior(title="Posterior prediction: Laplace-style kernel")


## Quantitative comparison

I compare kernels using the negative log marginal likelihood on the observed points. For the MSE, the posterior mean is interpolated back onto the full signal grid.

The two metrics answer slightly different questions: the marginal likelihood measures probabilistic model fit, while MSE measures point-prediction accuracy against the full signal.

In [ ]:
se_mean_on_data = np.interp(x_data, prediction_grid, gp_se.post_mean)
laplace_mean_on_data = np.interp(x_data, prediction_grid, gp_laplace.post_mean)

mse_se = np.mean((y_data - se_mean_on_data) ** 2)
mse_laplace = np.mean((y_data - laplace_mean_on_data) ** 2)

print(f"SE kernel       | NLL={nll_se:.2f} | MSE={mse_se:.2f}")
print(f"Laplace kernel  | NLL={nll_laplace:.2f} | MSE={mse_laplace:.2f}")


## Takeaways

- GP priors make kernel assumptions visible through sampled functions.
- The lengthscale controls smoothness, while the variance controls vertical scale.
- Marginal likelihood optimisation provides a principled way to tune hyperparameters.
- Posterior uncertainty grows away from observations and shrinks around observed data.
- The squared exponential kernel tends to produce smoother posterior means.
- The Laplace-style kernel can fit rougher local variation more closely, which can lead to lower MSE even when its marginal likelihood is worse.